# DP-SGD against a non-private baseline, on Criteo

Notebook 01 ran DP-SGD end to end and showed the pieces work. It had no
baseline, so it could not say what the privacy cost. This notebook adds the
missing arm: **plain SGD**, the same model trained by the same optimizer on the
same data with the noise and the clipping removed, and compares the two at
**equal steps** under a fixed budget of **ε = 3, δ = 1e-6**.

**On the numbering.** The notebook plan as notebook 01 laid it out reserved
"notebook 2" for the comparison against Private SpiderBoost and "notebook 3"
for the comparison against a non-private baseline, because the baseline loop
did not exist in the library yet (ADR-0005 requires it live there rather than
in a notebook). It does now — `dimma.algorithms.sgd`, commit `1bdbf87` — so
that comparison arrives first and is this notebook, **02**; the SpiderBoost
comparison moves later. Notebook 01 also left "notebook 2" holding several
obligations that belong to whichever notebook turns out to be the first real
comparison: fix an `R`, apply it to both arms and say where it came from, and
stop reporting a gap without repeats. They are discharged here — sections 1, 4
and 7 — whatever the file ends up numbered. This notebook does not modify
notebook 01; this paragraph is the reconciliation.

What is fixed across both arms, and why (ADR-0005, ADR-0002):

- the same per-sample loss, `per_sample_bce_loss`;
- the same model and the **same initialization**, `init_params(jax.random.key(0), d)`
  — one starting point, not one per arm;
- the same optimizer object, `updates.sgd(lr)`, with the arm's own tuned `lr`;
- the same split, the same `feature_norm_bound = R = 2.0`, the same evaluation
  set, the same threshold rule;
- the same **number of optimizer steps**. Privacy composes over steps, not over
  epochs, so steps are the axis on which the two arms are commensurable.

**The one difference that is not the privacy.** DP-SGD draws Poisson lots of
expected size `L`; plain SGD draws shuffled epochs of exact size `B`. That
difference is forced — Poisson sampling is what the amplification argument in
the accountant is about, and a baseline that used it would be paying for a
guarantee it does not make. So `L` and `B` are matched by name, not by
identity, and the realized lot size in the private arm varies around `L`. Every
other difference below is the privacy.

Two reading rules carried over from notebook 01:

- **The headline is PR-AUC**, read against the label base rate as its floor,
  with a confusion matrix beside it. Accuracy is useless at a 25% base rate.
  ROC-AUC is not reported.
- **A ranking metric cannot see the scale of `w`**, so log-loss is reported
  alongside throughout and is what section 4 selects on. A configuration can
  win on PR-AUC and lose badly on log-loss by overshooting the optimum along
  the same direction.

In [1]:
import time
from typing import NamedTuple

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from dimma.accounting.sampling import (
    PoissonGaussianSchedule,
    calibrate_noise_multiplier,
    poisson_gaussian_epsilon,
)
from dimma.algorithms.dp_sgd import train as dp_sgd
from dimma.algorithms.sgd import train as sgd
from dimma.core import updates
from dimma.datasets.criteo import load_criteo
from dimma.metrics.calibration import (
    calibration_ratio,
    expected_calibration_error,
    reliability_curve,
)
from dimma.metrics.decomposition import log_loss_decomposition
from dimma.metrics.scoring import log_loss, normalized_entropy
from dimma.models.logreg import forward, init_params
from dimma.models.losses import per_sample_bce_loss

# The budget, fixed before anything is fitted.
TARGET_EPSILON = 3.0
TARGET_DELTA = 1e-6

# The step budget. Both arms take exactly this many optimizer steps, and this
# is the composition count the noise multiplier is calibrated against.
STEPS = 10_000

# ADR-0012's R. Fixed a priori, never read off the data. Identical in both arms.
FEATURE_NORM_BOUND = 2.0

# The seed story: one initialization for both arms, one base seed for the
# stochastic parts, and a config index folded into it so every sweep run is
# reproducible from its position in the grid.
INIT_SEED = 0
BASE_SEED = 20260809

DEVICE = "gpu" if any(d.platform == "gpu" for d in jax.devices()) else "cpu"
print(f"jax devices: {jax.devices()}  ->  DEVICE = {DEVICE!r}")

jax devices: [CudaDevice(id=0)]  ->  DEVICE = 'gpu'


### On δ = 1e-6

The brief wrote the target as "10e-6". Read literally that is 1e-5; it is taken
here as **10⁻⁶ = 1e-6**, which is what the notation almost always means and what
notebook 01 used. It is also the standard pairing for this dataset: δ should sit
below 1/n, and the raw Criteo sample is 1,000,000 rows, so δ = 1e-6 = 1/n is the
conventional choice. Every ε below is at δ = 1e-6.

### Chart styling

One place for the palette, so every figure below reads as the same system. The
two series colours are categorical slots 1 and 2 of the reference palette; the
pair clears the colour-vision-deficiency and normal-vision separation floors
against a white surface. Hue never carries a series on its own — every series
is in the legend as well.

In [2]:
SERIES = {"plain SGD": "#2a78d6", "DP-SGD": "#eb6834"}   # categorical slots 1, 2
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED,
    "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": GRID, "grid.linewidth": 0.8,
    "lines.linewidth": 2.0, "lines.markersize": 6.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "font.size": 10,
})

## 1. The data, and a validation split that notebook 01 did not need

`load_criteo` is the only thing here that touches the dataset, and it records
what it did in `metadata["preprocessing"]` so a run carries its own provenance
(ADR-0008). All 39 features and the deterministic `seed=0` /
`test_fraction=0.2` split, as in notebook 01 — but with `R = 2.0` enforced,
which notebook 01 does not do. That difference is deliberate and it means the
two notebooks' numbers are not directly comparable: capping every row to
`‖x‖₂ ≤ 2` changes the data, and so the model. `R` is here because a comparison
needs it applied identically to both arms (ADR-0012); a smoke test of one
algorithm does not.

**What is new is a validation split.** Notebook 01 tuned nothing, so a two-way
train/test split was honest. This notebook selects hyperparameters for both
arms, and selecting on test would make the reported test numbers a fitted
quantity. So the 800,000-row training split is cut once, deterministically,
into **700,000 train / 100,000 validation**. The test split is never read until
section 6.

Both arms train on the same 700,000 rows and are selected on the same 100,000.
Note the consequence for the accountant: `n = 700,000`, so the sampling rate is
`q = L / 700_000`, and that is the `q` the calibration below uses.

**On `R`.** `feature_norm_bound` enforces `x -> x / max(1, ‖x‖/R)` per record,
after every fitted map. It does **not** enter DP-SGD's ε — sensitivity there
comes from clipping, `S = C`. But `R` changes the data, and so the model, which
is exactly why it has to be identical in both arms and fixed before looking at
the norms. An `R` read off the data is a fitted `L0` and voids the ε that rests
on it (ADR-0012).

In [3]:
split = load_criteo(features="all", preprocess=True, standardize=True,
                    download=False, seed=0,
                    device=DEVICE,
                    feature_norm_bound=FEATURE_NORM_BOUND)

n_all, n_features = split.x_train.shape

# One deterministic cut of the training split. Not a resample: a permutation
# with a fixed seed, held out once and never re-drawn.
holdout = np.random.default_rng(BASE_SEED).permutation(n_all)
val_index = jnp.asarray(holdout[:100_000])
train_index = jnp.asarray(holdout[100_000:])

X_TRAIN, Y_TRAIN = split.x_train[train_index], split.y_train[train_index]
X_VAL, Y_VAL = split.x_train[val_index], split.y_train[val_index]
X_TEST, Y_TEST = split.x_test, split.y_test

n_train = X_TRAIN.shape[0]
base_rate = float(Y_TEST.mean())

print(f"train {X_TRAIN.shape}   val {X_VAL.shape}   test {X_TEST.shape}")
print(f"label base rate: {float(Y_TRAIN.mean()):.4f} train, "
      f"{float(Y_VAL.mean()):.4f} val, {base_rate:.4f} test")
print(f"PR-AUC floor (a random ranking scores the base rate): {base_rate:.4f}")
print(f"arrays live on: {list(X_TRAIN.devices())}")
print()
print(f"enforced feature norm bound R = "
      f"{split.metadata['feature_norm_bound']}")
print()
print(split.metadata["preprocessing"])

criteo: 39 features. I1-I13: NaN filled with the train-split median, clipped below at 0, log1p. C1-C26: each category ID replaced by that category's relative frequency in the train split, with categories unseen in training encoded as 0.0. All 39 columns then standardized by the train-split mean and standard deviation. Every row then rescaled to l2 norm at most 2.0, one record at a time.


train (700000, 39)   val (100000, 39)   test (200000, 39)
label base rate: 0.2508 train, 0.2523 val, 0.2520 test
PR-AUC floor (a random ranking scores the base rate): 0.2520
arrays live on: [CudaDevice(id=0)]

enforced feature norm bound R = 2.0

39 features. I1-I13: NaN filled with the train-split median, clipped below at 0, log1p. C1-C26: each category ID replaced by that category's relative frequency in the train split, with categories unseen in training encoded as 0.0. All 39 columns then standardized by the train-split mean and standard deviation. Every row then rescaled to l2 norm at most 2.0, one record at a time.


### The caveats that travel with every ε below

Three accesses are outside every budget in this notebook, and they have to be
restated beside the number rather than filed away (ADR-0008, and the list in
`accounting/sampling.py`):

1. **The preprocessing statistics** — the medians, the category frequencies,
   the means and standard deviations — are fitted on the training split and
   accounted for in no budget. The validation rows were part of the split those
   statistics were fitted on, which is a second, smaller version of the same
   leak and one more reason the tuning below is not itself private.
2. **The hyperparameter search is never accounted.** Section 4 trains 54 models
   and reads validation loss off each one. Under the reported ε only the single
   selected run is covered. Making a search private is a different exercise.
3. **Evaluating on test is outside the loop by design.** Neither `train`
   returns metrics, precisely so the loop is not in a position to leak them.

There is also **one bit that does leak by design**: `cap_feature_norms` warns —
existence only, never a count — when some row's norm is non-finite, because for
that row the bound is believed rather than enforced (ADR-0012). Warnings are
not suppressed anywhere in this notebook. Nothing fired above, so no such row
exists in this split.

## 2. Shared machinery: one evaluator, one runner per arm

Everything that could differ between the arms by accident is written once here.
Both arms are trained by the same `fit_*` shape — fresh `init_params` from the
same key, the tuned `updates.sgd(lr)`, `STEPS` steps — and scored by the same
`evaluate_at`, which squashes the logits once and hands probabilities to
`dimma.metrics`. That package carries the reasoning for each number; only the
reason for reporting it here is below.

- **Log-loss** — the objective both arms descend, so selection and training
  agree on what better means. The CTR benchmarks fix it beside a ranking score,
  at 0.001 absolute (Zhu et al., *Open Benchmarking for CTR Prediction*,
  [arXiv:2009.05794](https://arxiv.org/abs/2009.05794)).
- **ECE**, at 15 equal-mass bins — because **per-example clipping, not the
  noise, is the documented cause of miscalibration under DP** (Zhang et al.,
  *A Closer Look at the Calibration of Differentially Private Learners*,
  [arXiv:2210.08248](https://arxiv.org/abs/2210.08248)), which is why the sweep
  below is over `C`. The estimator is biased upward and more so with more bins,
  so these compare only against each other.
- **Normalized entropy** and the **calibration ratio** — log-loss over the base
  rate's entropy, and observed clicks over predicted clicks. A bid built on a
  model at 0.9 overpays by about 11% (He et al., *Practical Lessons from
  Predicting Clicks on Ads at Facebook*, ADKDD 2014).
- **PR-AUC** — the ranking read, floored at the base rate. It stays a local
  helper: `dimma.metrics` offers no metric that reads rank alone, and
  `tests/metrics/test_package_surface.py` pins that absence.

The evaluator is notebook 01's, generalised off the test split so it can be
pointed at validation during the sweep, and extended with the pieces section 6
needs: a precision–recall curve, and a confusion matrix at an arbitrary
threshold rather than a hard-wired 0.5.

In [ ]:
N_BINS = 15    # notebook 01's bin count, so the two notebooks' ECEs compare
Y_VAL_NP = np.asarray(Y_VAL, dtype=np.float64)


class Eval(NamedTuple):
    """What one run is scored on, on whichever split it was pointed at.

    All three are threshold-free. The operating point arrives in section 6
    and nothing here depends on it.
    """

    loss: float
    pr_auc: float
    ece: float


def scores(params, x):
    """Logits for a whole array. `forward` is a single-example function."""
    return np.asarray(jax.vmap(forward, in_axes=(None, 0))(params, x))


def probabilities(logits):
    """The logits squashed once, so every number below reads one array.

    `dimma.metrics` takes probabilities and raises on anything else, so
    this is the notebook's single conversion point.
    """
    return np.asarray(jax.nn.sigmoid(jnp.asarray(logits)), dtype=np.float64)


def pr_curve(probs, y):
    """Precision and recall at every cut, plus average precision (PR-AUC).

    Sort by score descending and integrate precision over recall at every
    positive. Local by decision rather than by omission — see the note
    above.
    """
    order = np.argsort(-probs)
    hits = y[order]
    positives = hits.sum()
    tp = np.cumsum(hits)
    precision = tp / np.arange(1, len(hits) + 1)
    recall = tp / positives
    pr_auc = float((precision * hits).sum() / positives)
    return precision, recall, pr_auc, probs[order]


def confusion_at(probs, y, threshold):
    """(tn, fp, fn, tp) for `predict click when p >= threshold`."""
    pred = probs >= threshold
    truth = y > 0.5
    return (int((~pred & ~truth).sum()), int((pred & ~truth).sum()),
            int((~pred & truth).sum()), int((pred & truth).sum()))


def best_f1_threshold(probs, y):
    """The threshold maximising F1, and the F1 it reaches.

    Every distinct cut of the sorted scores is a candidate, so this is the
    exact maximiser over thresholds rather than a grid search.
    """
    precision, recall, _, sorted_probs = pr_curve(probs, y)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    best = int(np.argmax(f1))
    return float(sorted_probs[best]), float(f1[best])


def evaluate_at(params, x, y):
    """Score one parameter set on an arbitrary split. Guards divergence.

    The metrics raise on non-finite input rather than returning nan, so the
    guard stays in front of them: a diverged run has no scores, not bad
    ones.
    """
    logits = scores(params, x)
    if not np.all(np.isfinite(logits)):
        return Eval(float("nan"), float("nan"), float("nan"))
    probs, labels = probabilities(logits), np.asarray(y, dtype=np.float64)
    return Eval(loss=log_loss(probs, labels),
                pr_auc=pr_curve(probs, labels)[2],
                ece=expected_calibration_error(probs, labels, n_bins=N_BINS))


def show_confusion(confusion, title, threshold):
    """The 2x2 as a table — four numbers do not need a chart."""
    tn, fp, fn, tp = confusion
    print(f"{title}")
    print(f"                 predicted no    predicted click")
    print(f"  actual no      {tn:>12,}    {fp:>15,}")
    print(f"  actual click   {fn:>12,}    {tp:>15,}")
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    print(f"  precision {precision:.4f}   TPR (recall) {recall:.4f}   "
          f"(at threshold p >= {threshold:.4f})")

The two runners. They are deliberately the same function twice over, differing
only where ADR-0005 permits: the private one takes a clipping norm, a noise
multiplier and a JAX key for the noise, and it samples by Poisson.

`seed_index` is folded into `BASE_SEED` so each configuration gets its own
stochastic stream and the whole grid still reproduces from the notebook alone.
The **initialization is not affected by it** — that stays `jax.random.key(0)`
for every run in both arms.

In [5]:
def fresh_params():
    """The one starting point, shared by both arms and every configuration."""
    return init_params(jax.random.key(INIT_SEED), n_features)


def fit_sgd(*, learning_rate, batch_size, steps=STEPS, seed_index=0,
            params=None, rng=None):
    """Plain SGD: shuffled epochs, no clipping, no noise, no epsilon."""
    params = fresh_params() if params is None else params
    rng = np.random.default_rng([BASE_SEED, seed_index]) if rng is None else rng
    return sgd.train(
        per_sample_bce_loss, params, updates.sgd(learning_rate),
        X_TRAIN, Y_TRAIN, rng,
        steps=steps, batch_size=batch_size,
    ), rng


def fit_dp_sgd(*, learning_rate, expected_batch_size, clip_norm,
               noise_multiplier, steps=STEPS, seed_index=0, chunk=0,
               params=None, rng=None):
    """DP-SGD: Poisson lots, per-sample clipping to `clip_norm`, Gaussian noise."""
    params = fresh_params() if params is None else params
    rng = np.random.default_rng([BASE_SEED, seed_index]) if rng is None else rng
    key = jax.random.fold_in(jax.random.key(BASE_SEED + seed_index), chunk)
    return dp_sgd.train(
        per_sample_bce_loss, params, updates.sgd(learning_rate),
        X_TRAIN, Y_TRAIN, key, rng,
        steps=steps, expected_batch_size=expected_batch_size,
        clip_norm=clip_norm, noise_multiplier=noise_multiplier,
    ), rng

## 3. Calibrating the budget

`calibrate_noise_multiplier` runs the accountant backwards: given a target
`(ε, δ)` and the *shape* of the run, it returns the smallest noise multiplier
that meets the budget. It takes a builder — a function from a candidate
multiplier to the release schedules a run would have at that multiplier —
because a method may release on more than one schedule. Plain DP-SGD has
exactly one, so the builder returns a single-entry list.

The shape of the run is `(q, σ, T)` with

- `q = L / n`, the Poisson sampling rate, `n = 700,000` (the train split as cut
  in section 1);
- `T = STEPS = 10,000`, and this is the load-bearing constraint: **the step
  count the calibration assumes is the step count both the sweep and the final
  runs actually take.** Calibrating at 10,000 and then training for 20,000
  would report an ε the run does not have.

`clip_norm` does **not** appear. Sensitivity is `S = C` by construction, and
the noise scales with it (`σ·C`), so the multiplier is the same at every `C`
and the clipping norm is a free hyperparameter as far as the budget is
concerned. `R` does not appear either, for the reason in section 1. What does
appear is `L`, so the sweep needs one multiplier per expected lot size.

**The ε is RDP's** (ADR-0011, the `method="rdp"` default). PLD would report a
smaller number for the same run — at `q ≈ 0.001` and a tight budget, RDP costs
roughly a third more noise. The method is held fixed at `"rdp"` everywhere in
this notebook, which is the thing that has to be true for a comparison.

In [6]:
def sigma_for(expected_batch_size, steps=STEPS, epsilon=TARGET_EPSILON):
    """The smallest noise multiplier meeting (epsilon, delta) for this shape."""
    q = expected_batch_size / n_train
    return calibrate_noise_multiplier(
        lambda multiplier: [PoissonGaussianSchedule(q, multiplier, steps)],
        epsilon, TARGET_DELTA,
    )


DP_BATCH_GRID = [256, 1024]
SIGMA = {}

t0 = time.time()
for L in DP_BATCH_GRID:
    s = sigma_for(L)
    spent = poisson_gaussian_epsilon(L / n_train, s, STEPS, TARGET_DELTA)
    SIGMA[L] = s
    assert spent <= TARGET_EPSILON + 1e-9
    print(f"L={L:<6} q={L / n_train:.6f}  sigma={s:.6f}  "
          f"-> epsilon={spent:.6f} at delta={TARGET_DELTA} "
          f"over {STEPS:,} steps")
print(f"\ncalibrated in {time.time() - t0:.1f}s (method='rdp')")

L=256    q=0.000366  sigma=0.583168  -> epsilon=2.999998 at delta=1e-06 over 10,000 steps


L=1024   q=0.001463  sigma=0.699134  -> epsilon=2.999999 at delta=1e-06 over 10,000 steps

calibrated in 0.5s (method='rdp')


The reporting direction agrees with the calibrating one to floating point, and
lands at the target from below: the budget is met, not approached from above.

Note that σ barely moves between the two lot sizes while `q` moves by a factor
of four. That is the amplification working: a four-times larger lot is a
four-times larger `q`, and holding ε fixed asks for correspondingly more noise
per sample — but the *signal* also grows with the lot, so the ratio σ/L, which
is what the averaged gradient actually sees, improves by roughly the same
factor. Which of the two effects wins is not settled by that argument — it is
why the lot size has to be an axis of the sweep rather than a constant, and
section 4 finds the two lot sizes almost indistinguishable at this budget.

## 4. Hyperparameter search, both arms, selected on validation log-loss

Notebook 01 tuned nothing and said so. That is fine for a smoke test and fatal
for a comparison: an untuned baseline is an easy baseline, and an untuned
private arm understates what privacy can do. So **both** arms are swept, over
grids of comparable size, and each arm keeps its own best configuration.

- **Plain SGD** — learning rate `[0.01 … 3.0]` × batch size `[64, 256, 1024]`,
  6 × 3 = 18 runs.
- **DP-SGD** — learning rate `[0.03 … 10.0]` × clipping norm `[0.1, 0.3, 1.0]` ×
  expected lot size `[256, 1024]`, 6 × 3 × 2 = 36 runs. The clipping norm is the axis the baseline does not
  have, and it is the one that interacts with the learning rate: clipping to
  `C` scales every per-sample gradient down, so a small `C` needs a large `lr`
  to travel the same distance, and the useful configurations lie along a
  diagonal rather than at a point.

Both learning-rate grids deliberately run past the point where the arm gives
up, so that "the selected rate is interior to the grid" is something the sweep
shows rather than something it assumes. The two arms give up in different
places, and for a reason worth naming: the baseline's step is `lr` times an
unbounded gradient, and at `R = 2` the smoothness constant `L1 = 1.25` puts the
full-batch stability limit near `2/L1 = 1.6` — so `lr = 3.0` is the baseline's
losing end. The private arm clips every per-sample gradient to `C` first, which
caps the update at `lr · C` whatever the loss surface is doing, so it tolerates
much larger rates and needs a grid that reaches `10.0` to find its own edge.
That is also why `lr` and `C` cannot be swept independently: what matters to
the private arm is roughly their product.

Every run is `STEPS = 10,000` steps — the same step budget the final models
get, and the same one the calibration in section 3 assumed.

**The selection rule, stated once and applied identically to both arms: the
lowest validation log-loss.** Accuracy and ROC-AUC are not consulted, and
PR-AUC — reported everywhere below, and the headline — does not select.

Because clipping rather than the noise is what damages calibration (Zhang et
al., section 2), the ranking is the half of the model a private run is least
likely to have moved, so selecting on it chooses between configurations on the
axis that barely separates them. On this grid it does so visibly: the top few
DP-SGD rows are separated by 0.0001 in PR-AUC and by 0.19 in log-loss, and a
rule reading the first is picking among the second at random — reporting a
model whose calibration is an artifact of the tie rather than a cost of
privacy. A ranking metric cannot see the scale of `w`; log-loss can, and it is
the axis this notebook then goes on to measure the privacy cost along.

This is notebook 01's rule (`1ec52ed`), so both notebooks now select the same
way.

The whole search is unaccounted (section 1, caveat 2). The reported ε covers
the single selected DP-SGD run.

In [ ]:
SGD_LR_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
SGD_BATCH_GRID = [64, 256, 1024]

sgd_sweep = []
t0 = time.time()
for index, batch in enumerate(SGD_BATCH_GRID):
    for j, lr in enumerate(SGD_LR_GRID):
        seed_index = index * len(SGD_LR_GRID) + j
        params, _ = fit_sgd(learning_rate=lr, batch_size=batch,
                            seed_index=seed_index)
        ev = evaluate_at(params, X_VAL, Y_VAL)
        sgd_sweep.append(dict(lr=lr, batch=batch, loss=ev.loss,
                              pr_auc=ev.pr_auc, seed_index=seed_index))
        print(f"  B={batch:<6} lr={lr:<7} val log-loss={ev.loss:.4f}  "
              f"val PR-AUC={ev.pr_auc:.4f}")
print(f"\nplain SGD sweep: {len(sgd_sweep)} runs in {time.time() - t0:.0f}s")

In [ ]:
DP_LR_GRID = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
DP_CLIP_GRID = [0.1, 0.3, 1.0]

dp_sweep = []
t0 = time.time()
counter = 0
for L in DP_BATCH_GRID:
    for clip in DP_CLIP_GRID:
        for lr in DP_LR_GRID:
            params, _ = fit_dp_sgd(learning_rate=lr, expected_batch_size=L,
                                   clip_norm=clip, noise_multiplier=SIGMA[L],
                                   seed_index=counter)
            ev = evaluate_at(params, X_VAL, Y_VAL)
            dp_sweep.append(dict(lr=lr, clip=clip, batch=L, loss=ev.loss,
                                 pr_auc=ev.pr_auc, seed_index=counter))
            print(f"  L={L:<6} C={clip:<5} lr={lr:<7} "
                  f"val log-loss={ev.loss:.4f}  val PR-AUC={ev.pr_auc:.4f}")
            counter += 1
print(f"\nDP-SGD sweep: {len(dp_sweep)} runs in {time.time() - t0:.0f}s")

The top of each grid by validation log-loss, then the winners. Note how little
separates the top rows on PR-AUC and how much on log-loss — which is the
argument for the rule above, and a caution against reading any single ordering
off the ranking column. `nan` rows are runs that diverged — a
learning rate large enough to send the logits to infinity — and they are kept
in the table rather than filtered, because the shape of where a grid breaks is
part of what the grid measured.

In [ ]:
def best_of(sweep):
    """The lowest validation log-loss in the grid.

    Nan-safe: a diverged run has no score at all and cannot be selected.
    """
    finite = [row for row in sweep if np.isfinite(row["loss"])]
    return min(finite, key=lambda row: row["loss"])


def show_top(sweep, name, keys, count=5):
    finite = [r for r in sweep if np.isfinite(r["loss"])]
    ranked = sorted(finite, key=lambda r: r["loss"])[:count]
    diverged = len(sweep) - len(finite)
    print(f"{name}: top {count} of {len(sweep)} by validation log-loss"
          + (f"  ({diverged} diverged)" if diverged else ""))
    head = "  ".join(f"{k:>7}" for k in keys)
    print(f"  {head}  {'val log-loss':>13}  {'val PR-AUC':>11}")
    for row in ranked:
        body = "  ".join(f"{row[k]:>7}" for k in keys)
        print(f"  {body}  {row['loss']:>13.4f}  {row['pr_auc']:>11.4f}")
    print()


show_top(sgd_sweep, "plain SGD", ["batch", "lr"])
show_top(dp_sweep, "DP-SGD", ["batch", "clip", "lr"])

BEST_SGD = best_of(sgd_sweep)
BEST_DP = best_of(dp_sweep)
print("selected: the lowest validation log-loss in each arm, same rule both "
      "sides.")
print(f"selected  plain SGD : B={BEST_SGD['batch']}, lr={BEST_SGD['lr']}")
print(f"selected  DP-SGD    : L={BEST_DP['batch']}, C={BEST_DP['clip']}, "
      f"lr={BEST_DP['lr']}, sigma={SIGMA[BEST_DP['batch']]:.4f}")

The sweep as a picture: validation log-loss against learning rate, one line per
batch size for the baseline and one line per clipping norm (at the selected lot
size) for the private arm — the axis the rule above selects on, on one shared
scale so the two arms are read against each other. The dotted line is the
constant predictor, and the panel is cut just above it: a configuration that
scores worse than predicting the base rate has run off the top, which is the
same information as a `nan` row and takes no space to say.

In [ ]:
# What a model that ranks nothing scores. Also the denominator normalized
# entropy divides out, so it is the natural top of this axis.
CONST_VAL_LOSS = log_loss(np.full_like(Y_VAL_NP, float(Y_VAL_NP.mean())),
                          Y_VAL_NP)
best_loss = min(r["loss"] for r in sgd_sweep + dp_sweep
                if np.isfinite(r["loss"]))

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.5))

for batch in SGD_BATCH_GRID:
    rows = [r for r in sgd_sweep if r["batch"] == batch]
    left.plot([r["lr"] for r in rows], [r["loss"] for r in rows],
              marker="o", label=f"B = {batch}",
              alpha=0.45 + 0.55 * (batch == BEST_SGD["batch"]),
              color=SERIES["plain SGD"],
              linestyle=["-", "--", ":"][SGD_BATCH_GRID.index(batch)])

for clip in DP_CLIP_GRID:
    rows = [r for r in dp_sweep
            if r["clip"] == clip and r["batch"] == BEST_DP["batch"]]
    right.plot([r["lr"] for r in rows], [r["loss"] for r in rows],
               marker="o", label=f"C = {clip}",
               alpha=0.45 + 0.55 * (clip == BEST_DP["clip"]),
               color=SERIES["DP-SGD"],
               linestyle=["-", "--", ":"][DP_CLIP_GRID.index(clip)])

for axis, title, grid in (
        (left, "plain SGD", SGD_LR_GRID),
        (right, f"DP-SGD at L = {BEST_DP['batch']}, "
                f"eps = {TARGET_EPSILON:g}", DP_LR_GRID)):
    axis.axhline(CONST_VAL_LOSS, color=MUTED, linestyle=":", alpha=0.9)
    axis.annotate("constant predictor", xy=(grid[0], CONST_VAL_LOSS),
                  xytext=(0, -14), textcoords="offset points", color=MUTED,
                  fontsize=9)
    axis.set_xscale("log")
    axis.set_ylim(best_loss - 0.01, CONST_VAL_LOSS * 1.04)
    axis.set_xlabel("learning rate")
    axis.set_ylabel("validation log-loss")
    axis.set_title(title)
    axis.legend()
    axis.grid(alpha=0.5)

fig.suptitle("Validation log-loss over the grids; the same rule selects in "
             "both arms", color=INK)
fig.tight_layout()
plt.show()

## 5. Loss over steps, both arms, same axis

Neither `train` returns optimizer state, and that is deliberate: accepting one
would let a caller resume a run and replay its noise stream from the start.
Neither returns metrics either, because the loop is not in a position to claim
one. So a training curve has to be assembled from outside.

Notebook 01 assembled it from independent runs. This notebook instead calls
`train` in **40 chunks of 250 steps**, threading the parameters, the numpy
sampling stream and (for the private arm) a per-chunk JAX key through, and
evaluating between chunks. The inner loop is never reimplemented. What that
buys is a genuine trajectory — 40 points on one run rather than 40 separate
runs — and what it costs is worth stating:

- **The optimizer state resets each chunk.** `updates.sgd` carries only an
  update count, used solely by schedules; the learning rate here is constant,
  so the reset changes nothing. This trick would not be sound with momentum.
- **The shuffled-epoch stream restarts each chunk** in the baseline. With
  700,000 rows and chunks of 250 steps, no chunk completes an epoch anyway, so
  the practical effect is a reshuffle every 250 steps rather than every epoch.
- **The noise stream is a fresh key per chunk**, folded from the run key.
  Poisson sampling is memoryless and the Gaussian releases are independent
  either way, so the privacy analysis is unchanged: it is still 10,000
  Poisson-sampled Gaussian releases at multiplier σ, which is exactly what
  section 3 calibrated. The realized trajectory does differ from the
  monolithic 10,000-step run, so the model evaluated in section 6 is *this*
  run, not a re-run.

Both curves are read on the same two fixed evaluation sets: a fixed
100,000-row subsample of the training rows (so the curve shows the objective
being descended) and the full 100,000-row validation split (so it shows whether
that descent generalises). Same sets, same loss, both arms.

In [ ]:
CHUNKS = 40
CHUNK_STEPS = STEPS // CHUNKS
assert CHUNKS * CHUNK_STEPS == STEPS

# A fixed subsample of the training rows, drawn once, used by both arms.
curve_index = jnp.asarray(
    np.random.default_rng(BASE_SEED + 1).choice(n_train, 100_000,
                                                replace=False))
X_CURVE, Y_CURVE = X_TRAIN[curve_index], Y_TRAIN[curve_index]


def curve_point(params):
    """The four numbers recorded between chunks.

    Two arms, one trajectory each, so the calibration series is a real
    history rather than a set of independent runs read side by side.
    """
    train = evaluate_at(params, X_CURVE, Y_CURVE)
    val = evaluate_at(params, X_VAL, Y_VAL)
    return train.loss, val.loss, val.pr_auc, val.ece


def run_with_curve(arm):
    """`STEPS` steps in `CHUNKS` chunks, recording between them."""
    params = fresh_params()
    rng = np.random.default_rng([BASE_SEED, 0])
    record = [(0,) + curve_point(params)]
    for chunk in range(CHUNKS):
        if arm == "plain SGD":
            params, rng = fit_sgd(learning_rate=BEST_SGD["lr"],
                                  batch_size=BEST_SGD["batch"],
                                  steps=CHUNK_STEPS, params=params, rng=rng)
        else:
            params, rng = fit_dp_sgd(learning_rate=BEST_DP["lr"],
                                     expected_batch_size=BEST_DP["batch"],
                                     clip_norm=BEST_DP["clip"],
                                     noise_multiplier=SIGMA[BEST_DP["batch"]],
                                     steps=CHUNK_STEPS, chunk=chunk,
                                     params=params, rng=rng)
        record.append(((chunk + 1) * CHUNK_STEPS,) + curve_point(params))
    return params, record


FINAL, CURVE = {}, {}
for arm in ("plain SGD", "DP-SGD"):
    t0 = time.time()
    FINAL[arm], CURVE[arm] = run_with_curve(arm)
    last = CURVE[arm][-1]
    print(f"{arm:<10} {STEPS:,} steps in {time.time() - t0:5.0f}s   "
          f"train loss={last[1]:.4f}  val loss={last[2]:.4f}  "
          f"val PR-AUC={last[3]:.4f}  val ECE={last[4]:.4f}")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.5))

for arm in ("plain SGD", "DP-SGD"):
    steps_axis = [p[0] for p in CURVE[arm]]
    label = arm if arm == "plain SGD" else (
        f"DP-SGD (eps = {TARGET_EPSILON:g})")
    left.plot(steps_axis, [p[1] for p in CURVE[arm]], color=SERIES[arm],
              label=label)
    right.plot(steps_axis, [p[2] for p in CURVE[arm]], color=SERIES[arm],
               label=label)

for axis, title, ylabel in (
        (left, "Training loss (fixed 100k subsample)", "mean log-loss"),
        (right, "Validation loss (100k held out)", "mean log-loss")):
    axis.set_xlabel("optimizer steps")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.legend()
    axis.grid(alpha=0.5)

fig.suptitle(f"Same loss, same evaluation sets, same {STEPS:,} steps",
             color=INK)
fig.tight_layout()
plt.show()

Ranking quality and calibration over the same steps. These and the loss curves
above answer different questions — the loss can still be falling while the
ordering has stopped improving, because the remaining descent is rescaling `w`
rather than rotating it, and rescaling `w` is exactly what the right-hand panel
sees and the left-hand one does not.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.5))

for arm in ("plain SGD", "DP-SGD"):
    label = arm if arm == "plain SGD" else f"DP-SGD (eps = {TARGET_EPSILON:g})"
    steps_axis = [p[0] for p in CURVE[arm]]
    left.plot(steps_axis, [p[3] for p in CURVE[arm]], color=SERIES[arm],
              label=label)
    right.plot(steps_axis, [p[4] for p in CURVE[arm]], color=SERIES[arm],
               label=label)

left.axhline(base_rate, color=MUTED, linestyle=":", alpha=0.9)
left.annotate("PR-AUC floor (base rate)", xy=(0, base_rate), xytext=(0, 5),
              textcoords="offset points", color=MUTED, fontsize=9)

for axis, title, ylabel in (
        (left, "Ranking quality", "validation PR-AUC"),
        (right, f"Calibration ({N_BINS} equal-mass bins)",
         "validation ECE")):
    axis.set_xlabel("optimizer steps")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.legend()
    axis.grid(alpha=0.5)

fig.suptitle("The two halves of the score over the same step budget",
             color=INK)
fig.tight_layout()
plt.show()

## 6. The operating point, then the test split

A confusion matrix and a TPR are quantities at a threshold, and a threshold is
a choice. Made carelessly it becomes a third difference between the arms, so it
is made by **one rule applied identically to both models: the threshold that
maximises F1 on the validation split.**

Why that rule. At a 25% base rate the default cut of 0.5 on the probability is
badly placed — both models put most of their mass below it, so both report a
TPR near zero and the confusion matrices compare two models that never fire. F1
is the standard summary that forces a compromise between precision and TPR, it
needs no cost ratio invented for the occasion, and it is computable exactly:
every distinct cut of the sorted validation scores is a candidate, so the
maximiser is found rather than approximated.

The threshold is chosen **per model**, on validation, and then frozen before
the test split is read. Choosing it on test would make every number after it a
fitted quantity; choosing one shared threshold for both models would penalise
whichever model is differently scaled, which is not the same thing as being
worse. The rule is shared; the number it returns is each model's own.

In [ ]:
THRESHOLD, VAL_F1 = {}, {}
for arm in ("plain SGD", "DP-SGD"):
    probs = probabilities(scores(FINAL[arm], X_VAL))
    THRESHOLD[arm], VAL_F1[arm] = best_f1_threshold(probs, Y_VAL_NP)
    print(f"{arm:<10} validation-optimal threshold p >= {THRESHOLD[arm]:.4f}"
          f"   val F1 = {VAL_F1[arm]:.4f}")

Now, and only now, the test split.

In [ ]:
Y_TEST_NP = np.asarray(Y_TEST, dtype=np.float64)
TEST = {}

for arm in ("plain SGD", "DP-SGD"):
    probs = probabilities(scores(FINAL[arm], X_TEST))
    precision, recall, pr_auc, _ = pr_curve(probs, Y_TEST_NP)
    cm = confusion_at(probs, Y_TEST_NP, THRESHOLD[arm])
    tn, fp, fn, tp = cm
    TEST[arm] = dict(
        probs=probs, precision_curve=precision, recall_curve=recall,
        pr_auc=pr_auc, loss=log_loss(probs, Y_TEST_NP),
        ne=normalized_entropy(probs, Y_TEST_NP),
        ece=expected_calibration_error(probs, Y_TEST_NP, n_bins=N_BINS),
        cal_ratio=calibration_ratio(probs, Y_TEST_NP), confusion=cm,
        tpr=tp / (tp + fn), precision_at=tp / (tp + fp) if tp + fp else np.nan,
    )

# The threshold-free scores first: these are properties of the model.
CONST = np.full_like(Y_TEST_NP, base_rate)
print(f"{'':<12}{'log-loss':>10}{'NE':>8}{'ECE':>8}{'cal-ratio':>11}"
      f"{'PR-AUC':>9}")
print(f"{'constant':<12}{log_loss(CONST, Y_TEST_NP):10.4f}"
      f"{normalized_entropy(CONST, Y_TEST_NP):8.4f}"
      f"{expected_calibration_error(CONST, Y_TEST_NP, n_bins=N_BINS):8.4f}"
      f"{calibration_ratio(CONST, Y_TEST_NP):11.4f}{base_rate:9.4f}")
for arm in ("plain SGD", "DP-SGD"):
    r = TEST[arm]
    print(f"{arm:<12}{r['loss']:10.4f}{r['ne']:8.4f}{r['ece']:8.4f}"
          f"{r['cal_ratio']:11.4f}{r['pr_auc']:9.4f}")

# Then the operating point, which is a choice rather than a property.
print(f"\n{'':<12}{'TPR':>9}{'precision':>11}{'F1':>8}{'threshold':>11}")
for arm in ("plain SGD", "DP-SGD"):
    r = TEST[arm]
    f1 = 2 * r["precision_at"] * r["tpr"] / (r["precision_at"] + r["tpr"])
    print(f"{arm:<12}{r['tpr']:9.4f}{r['precision_at']:11.4f}{f1:8.4f}"
          f"{THRESHOLD[arm]:11.4f}")

print(f"\nDP-SGD at epsilon = {TARGET_EPSILON:g}, delta = {TARGET_DELTA}, "
      f"RDP accounting, over {STEPS:,} steps.")
print("plain SGD carries no epsilon: it is the unbounded-leakage end of the axis.")

In [ ]:
for arm in ("plain SGD", "DP-SGD"):
    show_confusion(TEST[arm]["confusion"], f"{arm} on the test split",
                   THRESHOLD[arm])
    print()

The precision–recall curves on one axes, with PR-AUC in the legend, the base
rate as the floor a random ranking would reach, and each model's chosen
operating point marked. The curve is the model's whole behaviour across
thresholds; the marker is the one point the confusion matrices above report.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 5))

for arm in ("plain SGD", "DP-SGD"):
    r = TEST[arm]
    # ~2000 points is plenty for a 200k-row curve and keeps the file small.
    stride = max(1, len(r["recall_curve"]) // 2000)
    label = (f"{arm} - PR-AUC {r['pr_auc']:.4f}" if arm == "plain SGD" else
             f"{arm} (eps = {TARGET_EPSILON:g}) - PR-AUC {r['pr_auc']:.4f}")
    axis.plot(r["recall_curve"][::stride], r["precision_curve"][::stride],
              color=SERIES[arm], label=label)
    axis.plot([r["tpr"]], [r["precision_at"]], marker="o", color=SERIES[arm],
              linestyle="none")

axis.axhline(base_rate, color=MUTED, linestyle=":", alpha=0.9)
axis.annotate(f"floor: base rate {base_rate:.4f}", xy=(0.02, base_rate),
              xytext=(0, 5), textcoords="offset points", color=MUTED,
              fontsize=9)
axis.set_xlabel("recall (TPR)")
axis.set_ylabel("precision")
axis.set_xlim(0, 1)
axis.set_title("Test precision-recall; markers are the validation-F1 threshold")
axis.legend(loc="upper right")
axis.grid(alpha=0.5)
fig.tight_layout()
plt.show()

The same two operating points as matrices, side by side and on a shared count
scale, so the difference is read in records rather than in rates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
labels = ["no click", "click"]
shared_max = max(max(TEST[arm]["confusion"]) for arm in TEST)

for axis, arm in zip(axes, ("plain SGD", "DP-SGD")):
    tn, fp, fn, tp = TEST[arm]["confusion"]
    grid = np.array([[tn, fp], [fn, tp]])
    axis.imshow(grid, cmap="Blues", vmin=0, vmax=shared_max)
    for i in range(2):
        for j in range(2):
            axis.text(j, i, f"{grid[i, j]:,}", ha="center", va="center",
                      fontsize=12,
                      color="white" if grid[i, j] > shared_max * 0.55 else INK)
    axis.set_xticks([0, 1], [f"predicted\n{lab}" for lab in labels])
    axis.set_yticks([0, 1], [f"actual\n{lab}" for lab in labels])
    axis.set_title(f"{arm}   (threshold {THRESHOLD[arm]:.3f})")
    for spine in axis.spines.values():
        spine.set_visible(False)
    axis.tick_params(length=0)
    axis.grid(False)

fig.suptitle(f"Test split, {len(Y_TEST_NP):,} rows, shared colour scale",
             color=INK)
fig.tight_layout()
plt.show()

### Calibration, measured rather than inferred

Everything above is either a ranking read or a quantity at a threshold. The two
below are neither. Zhang et al. measured their finding on vision and language
models; this is where it gets checked on tabular CTR, on two arms at the same
budget rather than on one arm against a noiseless control as notebook 01 did.

In [ ]:
RELIABILITY = {arm: reliability_curve(TEST[arm]["probs"], Y_TEST_NP,
                                     n_bins=N_BINS)
               for arm in ("plain SGD", "DP-SGD")}

points = np.concatenate([np.concatenate([c.mean_predicted, c.mean_observed])
                         for c in RELIABILITY.values()])
pad = 0.05 * (points.max() - points.min())
lo, hi = points.min() - pad, points.max() + pad

fig, axis = plt.subplots(figsize=(5.5, 5.5))
axis.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1, color=MUTED,
          zorder=1, label="perfectly calibrated")

# Identity is in the legend rather than at the end of each line: the two
# curves are expected to sit almost on top of each other and on the diagonal,
# which is where a direct label stops being readable.
for arm in ("plain SGD", "DP-SGD"):
    curve = RELIABILITY[arm]
    name = arm if arm == "plain SGD" else f"DP-SGD (eps = {TARGET_EPSILON:g})"
    axis.plot(curve.mean_predicted, curve.mean_observed, marker="o",
              markersize=7, linewidth=2, color=SERIES[arm], zorder=2,
              markeredgecolor="white", markeredgewidth=1.5,
              label=f"{name}    ECE {TEST[arm]['ece']:.4f}")

axis.set_xlim(lo, hi)
axis.set_ylim(lo, hi)
axis.set_aspect("equal")
axis.set_xlabel("mean predicted probability")
axis.set_ylabel("observed click rate")
axis.set_title(f"Test reliability, {N_BINS} equal-mass bins", color=INK)
axis.grid(alpha=0.5)
axis.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

for arm in ("plain SGD", "DP-SGD"):
    curve = RELIABILITY[arm]
    worst = curve.gap[int(np.argmax(np.abs(curve.gap)))]
    print(f"{arm:<10} {len(curve.count)} bins kept, "
          f"{curve.count.min():,}-{curve.count.max():,} records each, "
          f"largest gap {worst:+.4f} (predicted - observed)")

The same thing as numbers, and the reason this notebook can settle its own
central claim instead of inferring it:
`log-loss = calibration - resolution + uncertainty + residual`, at the same
bins. Under equal-mass binning the resolution term is rank-based, so it is
what a ranking score sees and the calibration term is precisely what it
cannot. `uncertainty` is a property of the test labels and is identical across
the arms by construction. A log-loss gap that sits in `calibration` is the
claim the closing section makes; one that sits in `resolution` falsifies it.

In [ ]:
DECOMPOSITION = {arm: log_loss_decomposition(TEST[arm]["probs"], Y_TEST_NP,
                                            n_bins=N_BINS)
                 for arm in ("plain SGD", "DP-SGD")}

print(f"{'':<12}{'calibration':>13}{'resolution':>12}{'uncertainty':>13}"
      f"{'residual':>10}{'log-loss':>10}")
for arm in ("plain SGD", "DP-SGD"):
    d = DECOMPOSITION[arm]
    print(f"{arm:<12}{d.calibration:13.4f}{d.resolution:12.4f}"
          f"{d.uncertainty:13.4f}{d.residual:10.4f}{d.total:10.4f}")

baseline, private = DECOMPOSITION["plain SGD"], DECOMPOSITION["DP-SGD"]
print(f"\n{'private - baseline':<12}")
print(f"  calibration {private.calibration - baseline.calibration:+.4f}   "
      f"resolution {private.resolution - baseline.resolution:+.4f}   "
      f"residual {private.residual - baseline.residual:+.4f}   "
      f"->  log-loss {private.total - baseline.total:+.4f}")

## 7. Does the gap survive a change of seed?

Notebook 01 ran one seed and flagged the absence of error bars as the next
notebook's problem. This notebook *is* the comparison, so the gap needs at
least a sense of its own noise. The selected configuration of each arm is
re-run at three seeds — the initialization stays `jax.random.key(0)`, as ADR-0005
requires; what changes is the sampling stream and, in the private arm, the
noise. These are monolithic 10,000-step runs, not the chunked ones from section
5, so the two sets of numbers also say something about the chunking. Read
that comparison carefully: the chunked and monolithic runs are different
trajectories, not the same run computed twice, so a difference between them is
not an error — it is one more draw.

Three seeds is a spread, not a confidence interval. It is reported as min/max.

In [ ]:
REPEATS = {}
t0 = time.time()
for arm in ("plain SGD", "DP-SGD"):
    values = []
    for seed_index in (0, 1, 2):
        if arm == "plain SGD":
            params, _ = fit_sgd(learning_rate=BEST_SGD["lr"],
                                batch_size=BEST_SGD["batch"],
                                seed_index=1000 + seed_index)
        else:
            params, _ = fit_dp_sgd(learning_rate=BEST_DP["lr"],
                                   expected_batch_size=BEST_DP["batch"],
                                   clip_norm=BEST_DP["clip"],
                                   noise_multiplier=SIGMA[BEST_DP["batch"]],
                                   seed_index=1000 + seed_index)
        values.append(evaluate_at(params, X_TEST, Y_TEST))
    REPEATS[arm] = values
    losses = [v.loss for v in values]
    prs = [v.pr_auc for v in values]
    eces = [v.ece for v in values]
    print(f"{arm:<10} test log-loss {np.mean(losses):.4f} "
          f"[{min(losses):.4f}, {max(losses):.4f}]   "
          f"PR-AUC {np.mean(prs):.4f} [{min(prs):.4f}, {max(prs):.4f}]   "
          f"ECE {np.mean(eces):.4f} [{min(eces):.4f}, {max(eces):.4f}]   "
          f"(chunked run: {TEST[arm]['loss']:.4f} / "
          f"{TEST[arm]['pr_auc']:.4f} / {TEST[arm]['ece']:.4f})")


def gap_of(field):
    """Baseline minus private: at the means, and at the two extremes."""
    base = [getattr(v, field) for v in REPEATS["plain SGD"]]
    private = [getattr(v, field) for v in REPEATS["DP-SGD"]]
    return (float(np.mean(base)) - float(np.mean(private)),
            min(base) - max(private), max(base) - min(private))


print()
for field, name in (("pr_auc", "PR-AUC"), ("loss", "log-loss"),
                    ("ece", "ECE")):
    mean_gap, low, high = gap_of(field)
    print(f"{name:>9} gap (baseline - private): mean {mean_gap:+.4f}, "
          f"range [{low:+.4f}, {high:+.4f}] over 3 seeds")
print(f"\n({time.time() - t0:.0f}s)")

## What this shows, and what it does not

> **Stale until re-run.** Every number quoted below is read off the run
> committed in `17d18bb`, which selected on validation PR-AUC with a log-loss
> tie-break. The rule is now validation log-loss, so the selected
> configurations — and everything sections 5, 6 and 7 report — have to be
> re-run before this section is read. What is said here about *what is
> measured* is current; the figures it quotes are not.

**The comparison.** Both arms descend the same loss on the same 700,000 rows
with the same initialization, the same optimizer, the same `R = 2.0` and the
same 10,000 optimizer steps, each at its own configuration chosen by the same
rule on the same validation split. The private arm additionally clips every
per-sample gradient to `C` and adds Gaussian noise at the multiplier the
accountant returned for ε = 3, δ = 1e-6 over exactly those 10,000 steps. The
only other difference is the sampling scheme — Poisson lots against shuffled
epochs — and that difference is forced by the amplification argument rather
than chosen.

**What the privacy cost, in the metrics that matter here.** The headline is
that on *this* problem, at this budget, **the ranking cost of ε = 3 is not
measurable and the calibration cost is.**

The ranking side first. Test PR-AUC is 0.4462 for the baseline and 0.4456 for
the private arm — a gap of 0.0006, against a floor of 0.2520. Read against that
floor rather than against 1.0, the private model keeps **99.7% of the
baseline's lift**. And the gap does not survive contact with section 7: over
three seeds the means are 0.4464 for the baseline and 0.4485 for the private
arm, so the sign flips, and the spread within each arm (about 0.001) is larger
than the difference between them. The honest statement is that the two arms
rank the test set equally well and this experiment cannot separate them. The
precision–recall curves in section 6 say the same thing by lying on top of each
other.

The calibration side is different, and it is where the privacy shows up. Test
log-loss is 0.5123 against 0.5402, a gap of 0.028 — and the baseline's own
three-seed range is 0.0001 wide, so unlike the PR-AUC gap this one is far
outside the noise. Clipping and noise leave `w` in a systematically different
place along the same direction: good enough to order the rows identically,
scaled wrongly enough to be visibly overconfident about it. That is exactly the
asymmetry notebook 01 warned about when it insisted on reporting log-loss
beside a ranking metric, and here it is the whole difference between the arms.
That attribution is no longer inferred from a flat PR-AUC: the decomposition at
the end of section 6 splits the log-loss gap into its calibration and
resolution halves directly, and the reliability diagram beside it is the same
statement as a picture.

In records, at the F1 operating point: the private model catches **468 fewer
clicks** (33,375 against 33,843 of 50,400) and raises **1,462 fewer false
alarms** (53,033 against 54,495). TPR falls from 0.6715 to 0.6622, precision
rises from 0.3831 to 0.3862, and F1 is 0.4879 for both to four decimals. It is
a slightly more conservative model, not a worse one.

**Why the cost is this small, and where that stops being true.** 700,000 rows,
39 features, a convex model, `q ≈ 0.0004`, and a fairly generous ε. The signal
this problem needs is a 40-dimensional direction estimated from lots of
hundreds; the noise the accountant demands at ε = 3 is small next to the
gradient noise already there from sampling. None of that survives a move to a
model with millions of parameters, a tighter ε, or a dataset where `n/d` is not
five figures. The number to carry away is the shape of the finding — ranking
survives, calibration does not — rather than "privacy costs 0.0006 PR-AUC".

**What the loss curves show.** Section 5's curves are the same run seen over
time. Both arms do essentially all of their descent in the first few hundred
steps and are flat from about step 500 onward, which is why the step budget is
not the binding constraint here: the remaining 9,500 steps buy almost nothing
and, in the private arm, cost ε. The private curve sits about 0.028 above the
baseline's for the whole run — the calibration gap is present from the start,
not accumulated, which the ECE panel now says outright rather than by proxy —
and it visibly oscillates in a band of roughly ±0.01 where
the baseline's is a flat line. That band is the injected noise arriving in the
gradient. Compare the loss panels against the ranking and calibration panels
before concluding anything: PR-AUC is flat and identical for both arms from step 500 while the
loss curves stay apart, which is the same story the final numbers tell.

**What the threshold rule buys and costs.** Reporting a TPR at all requires a
cut, and the validation-F1 rule gives one that both models are held to
identically. It is not the only defensible rule — a deployment with a known
cost ratio between a missed click and a wasted impression should use that ratio
instead, and the PR curve in section 6 is the object to read it off. The TPRs
reported here are not a property of the models alone; they are a property of
the models at F1's compromise.

Not shown, and not claimed:

- **Everything in section 1's caveat list.** The preprocessing statistics are
  fitted on the training split and are in no budget; the 54-run hyperparameter
  search is unaccounted, so the reported ε covers only the selected run; the
  test evaluation is outside the loop by design. A genuinely end-to-end private
  pipeline would have to pay for all three, and would post a worse number than
  this one at the same ε.
- **Three seeds is not an error bar.** Section 7 reports a spread, which is
  enough to say whether the gap is larger than run-to-run wobble and not enough
  to put an interval on it.
- **One budget.** ε = 3 only, so this notebook says what privacy costs *at
  that budget* and nothing about how that cost moves as the budget does. Tuned
  PR-AUC as a function of ε is a different experiment, and not one a single-ε
  run can be extrapolated into: the selected configuration is itself a function
  of ε, and the private arm's best `C` and learning rate at ε = 0.1 need not be
  the ones section 4 chose here.
- **The ε is RDP's** (ADR-0011). PLD would report a smaller ε for the same run,
  which is to say the private arm here is being charged more than it strictly
  spends. Held fixed across the comparison, which is the property that matters.
- **`logreg_bce_constants(R, has_bias=True)` was not used as a hyperparameter
  source.** At `R = 2` it gives `L0 ≈ 2.236`, `L1 = 1.25` and a theorem step of
  `η = 0.4`. Those are worst-case constants; the sweep in section 4 searched
  over learning rates empirically instead, and reported what it found. Swapping
  the theorem's step in silently would be reporting a guarantee the tuning
  already broke.
- **One dataset, one model.** Logistic regression on Criteo. A gap measured on
  a convex problem with 39 features says nothing directly about a deep model.